In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style="whitegrid")

FINAL_OUTPUT_PATH = "full_llm_output.csv"
JSONL_LOG_PATH = "full_api_log.jsonl"
GT_PATH = os.path.join("..", "data", "full_ground_truth.csv")
FIG_PATH = os.path.join("..", "figures", "fig2_comparison.png")

df_output = pd.read_csv(FINAL_OUTPUT_PATH)
df_gt = pd.read_csv(GT_PATH) if os.path.exists(GT_PATH) else pd.DataFrame()
print(f"Loaded {len(df_output)} full-run rows.")
df_output.head()

In [ ]:
if not df_gt.empty and "ground_truth" in df_gt.columns:
    df_eval = pd.merge(df_gt[["id", "ground_truth"]], df_output[["id", "llm_output"]], on="id", how="inner")
    exact_match = (df_eval["ground_truth"].astype(str) == df_eval["llm_output"].astype(str)).mean()
    gt_len = df_eval["ground_truth"].astype(str).map(len)
    llm_len = df_eval["llm_output"].astype(str).map(len)
    stat, p_value = stats.wilcoxon(gt_len, llm_len) if len(df_eval) >= 2 else (np.nan, np.nan)
    print(f"Exact match: {exact_match:.4f}")
    print(f"Wilcoxon p-value: {p_value}")
else:
    print("Chưa có full_ground_truth.csv — bỏ qua test thống kê.")

In [ ]:
log_records = []
if os.path.exists(JSONL_LOG_PATH):
    with open(JSONL_LOG_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                log_records.append(json.loads(line))

df_logs = pd.DataFrame(log_records) if log_records else pd.DataFrame()
total_cost = df_logs["cost_usd"].sum() if "cost_usd" in df_logs.columns else 0.0
print(f"Total cost: ${total_cost:.6f} USD")

In [ ]:
if "df_eval" in globals() and not df_eval.empty:
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    sns.kdeplot(df_eval["ground_truth"].astype(str).map(len), label="ground_truth", ax=ax)
    sns.kdeplot(df_eval["llm_output"].astype(str).map(len), label="llm_output", ax=ax)
    ax.legend()
    ax.set_title("Full run: GT vs LLM length distribution")
    os.makedirs(os.path.dirname(FIG_PATH), exist_ok=True)
    fig.savefig(FIG_PATH, bbox_inches="tight")
    print("Saved", FIG_PATH)
    plt.show()